### **ACID - PART 3 - Control image quality**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2025/12/12


## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [1]:
# Import required modules
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
from utils.listdirNHF import listdirNHF
from utils.get_defaults import default_file_name
# from utils.mksubdir import mk_subdir
# from image_processing.extract_metadata import extract_bioio_scene_metadata
# from image_processing.name_metadata import extract_name_metadata
# from image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
# from utils.open_image import bioio_open_image
# from utils.save_image import tifffile_save_ometiff
# from image_processing.save_metadata import save_xml_string



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [2]:
# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"

# indicate the path to the directory storing the fields of view
fov_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov"

# # indicate the path to the directory where outputs will be saved
# output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\develop\251205_train_test_split"


# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestap of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
metadata_file_name = "default"


# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"
# # --- parameters to select the train set ---
# indicate the name of the column indicating whether the row belongs to train or test set
is_train_column="is_train"

# indicate the value signalling that a row (aka a field of view) belongs to the train set
train_val=1

# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv"
default_metadata_file_exclude = None

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
metadata_default_separator = '_'

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
metadata_default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True


# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = '_'

# project
project_name = "ACID"

# ignore indexes when saving pandas dataframes as csv files
save_csv_index = False # if False, the index will not be saved as a separate column in the csv file

# metadata saving date format
metadata_date_format = '%Y%m%d'

# metadata savingword
metadata_savingword = "metadata"

# metadata file suffix
metadata_file_suffix = f"part{save_file_name_separator}3.csv"

# hyperparameters saving date format
hyperparameters_date_format = '%Y%m%d-%H%M%S'

# hyperparameters savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}3.csv"


# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = True # if the secondary output directory already exists, do not raise an error



### Create secondary output directory if it doesn't exist - this directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [3]:
# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)


#### Open the metadata dataframe - this is expected to be the output of part 2

Run the following cell.

Don't modify the following cell.

In [4]:

# check if using the default metadata data frame (the most recently saved)
if metadata_file_name==None or metadata_file_name.lower()=="default" or metadata_file_name=="":
    
    # import target files in the metadata_directory
    metadata_files = listdirNHF(metadata_directory,
                                target=default_metadata_file_target,
                                exclude=default_metadata_file_exclude)
    
    # get the default metadata file
    metadata_file_name = default_file_name(file_list=metadata_files,
                                           from_file_name=metadata_from_file_name,
                                           directory_path=metadata_directory,
                                           separator=metadata_default_separator,
                                           date_position=metadata_default_date_position,
                                           date_format=metadata_default_date_format,
                                           reverse=metadata_default_reverse)
    
    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df



using 20251211_ACID_metadata_part_2.csv as default metadata file


#### Select train data set - NOTE: the metadata_df is updated into a metadata_df which does not contain the test data

Run the following cell.

Don't modify the following cell.

In [5]:
# Select only the train set and update metadata_df
metadata_df = metadata_df[metadata_df[is_train_column] == train_val]

# assert proper selection of train set
assert metadata_df.shape[0] > 0, "No rows in metadata_df belong to the train set."
assert all(metadata_df[is_train_column] == train_val), "Not all rows in metadata_df belong to the train set."

# Display the metadata dataframe
metadata_df

,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,infectious_organism,...,size_c,size_z,size_y,physical_size_y,size_x,physical_size_x,dims_order,int_well,treatment,is_train
0,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A1,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
4,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A5,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
5,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A6,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
7,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,B7,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B7...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1166,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,F3,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_F3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,8,uninfected_JNJ_A07,1
1170,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G2,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,8,uninfected_JNJ_A07,1
1172,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G4,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,8,uninfected_JNJ_A07,1
1174,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G6,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,8,uninfected_JNJ_A07,1


#### Test PLLS computation

In [6]:
from image_quality_control.test_compute_plls import run_all_tests

run_all_tests()

# import numpy as np

# # ----------------------------------------------------
# #  IMPORT YOUR FUNCTION
# # ----------------------------------------------------
# # If compute_plls is in a file compute_plls.py in the same folder:
# # from compute_plls import compute_plls
# #
# # If it's elsewhere, adjust PYTHONPATH or insert the path manually:
# # import sys; sys.path.append("path/to/folder")
# # from compute_plls import compute_plls

# from image_quality_control.measure_plls import compute_plls


# # ----------------------------------------------------
# #  HELPER: run a test and print a nice summary
# # ----------------------------------------------------
# def run_test(name, func):
#     print(f"▶ {name} ... ", end="")
#     try:
#         func()
#         print("OK")
#     except AssertionError as e:
#         print("FAIL")
#         raise e


# # ----------------------------------------------------
# #  TEST 1 — deterministic PLLS on a simple gradient
# # ----------------------------------------------------
# def test_gradient_image():
#     img = np.tile(np.linspace(0, 1, 128), (128, 1))
#     slope = compute_plls(img)
#     print(type(slope))
#     # For a smooth gradient, high frequencies are nearly zero
#     # which gives a quite steep negative log–log slope.
#     assert np.isfinite(slope)
#     assert slope < -1  # Should be strongly negative.


# # ----------------------------------------------------
# #  TEST 2 — noise should give a flatter slope
# # ----------------------------------------------------
# def test_noise_image():
#     rng = np.random.default_rng(123)
#     img = rng.normal(0, 1, (128, 128))
#     slope = compute_plls(img)
    
#     # White noise has a roughly flat power spectrum → slope ≈ 0
#     assert np.isfinite(slope)
#     assert -0.5 < slope < 0.5


# # ----------------------------------------------------
# #  TEST 3 — constant image → PLLS cannot be computed
# # ----------------------------------------------------
# def test_constant_image():
#     img = np.ones((128, 128))
#     slope = compute_plls(img)

#     # Most implementations return NaN for fully flat FFT power
#     assert np.isnan(slope)


# # ----------------------------------------------------
# #  TEST 4 — axis slicing on z-stacks
# # ----------------------------------------------------
# def test_axis_slicing():
#     # A stack of 3 slices, each with different structure
#     img = np.zeros((3, 64, 64))

#     img[0] = np.random.random((64, 64))      # noise
#     img[1] = np.linspace(0, 1, 64).reshape(64,1)  # gradient
#     img[2] = 5                                # constant

#     slopes = compute_plls(img, axis=0)

#     assert len(slopes) == 3
#     assert np.isfinite(slopes[0])             # noise slice → finite
#     assert np.isfinite(slopes[1])             # gradient → finite
#     assert np.isnan(slopes[2])                # constant → NaN


# # ----------------------------------------------------
# #  TEST 5 — check verbosity does not break anything
# # ----------------------------------------------------
# def test_verbose_mode():
#     img = np.random.random((64, 64))
#     slope = compute_plls(img, verbose=2)   # Should print info
#     assert np.isfinite(slope)


# # img = np.tile(np.linspace(0, 1, 128), (128, 1))
# # slope, _, freqs, power = compute_plls(img, return_details=True)

# # print("Any zeros in power?", np.any(power == 0))
# # print("Any negative values?", np.any(power < 0))
# # print("Any NaN?", np.any(np.isnan(power)))
# # print("Any inf?", np.any(np.isinf(power)))


# # ----------------------------------------------------
# #  RUN THE TEST SUITE
# # ----------------------------------------------------
# run_test("Gradient image", test_gradient_image)
# run_test("Noise image", test_noise_image)
# run_test("Constant image", test_constant_image)
# run_test("Z-stack axis slicing", test_axis_slicing)
# run_test("Verbose mode", test_verbose_mode)

# print("\nAll tests passed ✔️")


▶ test_gradient_image ... OK
▶ test_noise_image ... OK
▶ test_constant_image ... OK
▶ test_axis_slicing ... OK
▶ test_verbose_mode ... [PLLS] Computing PLLS for single 2D image.
[PLLS-2D] Using Numba-accelerated radial binning.
[PLLS-2D] slope = -0.0661, intercept = 5.9569
OK
▶ test_tiny_images ... OK
▶ test_nan_image ... OK
▶ test_single_slice_zstack ... OK
▶ test_dc_only_image ... OK
▶ test_extreme_aspect_ratio ... OK
▶ test_invalid_input ... OK
▶ test_negative_values ... OK
▶ test_large_image ... OK
▶ test_multichannel_image ... OK
▶ test_single_pixel_zstack ... OK
▶ test_min_max_intensity ... OK
▶ test_plotting_mode ... ERROR
'bool' object is not iterable


TypeError: 'bool' object is not iterable

#### Add train-test split column to metadata dataframe and save the results

Run the following cell.

Don't modify the following cell.

In [ ]:


# # save the metadata dataframe with train test split column
# # save metadata dataframe as a csv file
# metadata_saving_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
# metadata_df_w_split.to_csv(os.path.join(metadata_directory,metadata_saving_name), index=save_csv_index)

# # display metadata dataframe with train test split column
# metadata_df_w_split


### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# # # collect hyperparameters in a dictionary

# hyperparameter_dict = {

# 'test_size':test_size,
# 'metadata_directory':metadata_directory,
# 'plate_layout_directory':plate_layout_directory,
# 'output_directory':output_directory,
# 'metadata_file_name':metadata_file_name,
# 'plate_layout_file_name':plate_layout_file_name,
# 'is_train_column':is_train_column,
# 'train_val':train_val,
# 'test_val':test_val,
# 'train_test_split_kwargs':train_test_split_kwargs,
# 'concat_kwargs':concat_kwargs,
# 'default_metadata_file_target':default_metadata_file_target,
# 'default_metadata_file_exclude':default_metadata_file_exclude,
# 'default_platelayout_file_target':default_platelayout_file_target,
# 'default_platelayout_file_exclude':default_platelayout_file_exclude,
# 'metadata_from_file_name':metadata_from_file_name,
# 'platelayout_from_file_name':platelayout_from_file_name,
# 'metadata_default_separator':metadata_default_separator,
# 'platelayout_default_separator':platelayout_default_separator,
# 'metadata_default_date_position':metadata_default_date_position,
# 'platelayout_default_date_position':platelayout_default_date_position,
# 'metadata_default_date_format':metadata_default_date_format,
# 'platelayout_default_date_format':platelayout_default_date_format,
# 'metadata_default_reverse':metadata_default_reverse,
# 'platelayout_default_reverse':platelayout_default_reverse,
# 'save_file_name_separator':save_file_name_separator,
# 'project_name':project_name,
# 'metadata_date_format':metadata_date_format,
# 'metadata_savingword':metadata_savingword,
# 'metadata_file_suffix':metadata_file_suffix,
# 'hyperparameters_date_format':hyperparameters_date_format,
# 'hyperparameters_savingword':hyperparameters_savingword,
# 'hyperparameters_file_suffix':hyperparameters_file_suffix,
# 'secondary_output_directory':secondary_output_directory,
# 'exist_ok':exist_ok
# }



# # transform the hyperparameter_dict in a pandas series
# hyperparameter_series = pd.Series(hyperparameter_dict)

# # save hyperparamters
# hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
# hyperparameter_series.to_csv(os.path.join(secondary_output_directory,hyperparameter_saving_name), index=save_csv_index)

